# Using Causal Reasoning to Test Scientific Software

## Michael Foster and Sylvia Whittle

m.foster@sheffield.ac.uk &nbsp;&nbsp;&nbsp;&nbsp;&nbsp; sylvia.whittle@sheffield.ac.uk

# Motivating Example: Covasim
- Stochastic agent-based simulator for performing COVID-19 analyses
- Used to inform research studies and policy decisions in multiple countries including the US, UK, and Australia
- Important to make sure the model is working properly

In [ ]:
import covasim as cv

sim = cv.Sim(location="UK", beta=0.016, pop_type="hybrid", verbose=0)
sim.run()
sim.summarize()

# A basic regression test

In [ ]:
EXPECTED_VALUE = 10800


def cumulative_infections(sim: cv.Sim):
    return int(sim.results["cum_infections"][-1])


def test_run_uk():
    sim = cv.Sim(location="UK", beta=0.016, pop_type="hybrid", verbose=0)
    sim.run()
    final_infections = cumulative_infections(sim)
    assert final_infections == EXPECTED_VALUE, f"{final_infections} != {EXPECTED_VALUE}"
    print(f"{final_infections} == {EXPECTED_VALUE}")


test_run_uk()

# Is the model working correctly?
If it is, we should see 10,800 every time we run the simulator...

In [ ]:
msim = cv.MultiSim(cv.Sim(location="UK", beta=0.016, pop_type="hybrid", verbose=0))
msim.run(n_runs=5)
for i, sim in enumerate(msim.sims):
    print(f"Run {i+1}: {cumulative_infections(sim)} total infections")

**So which is correct?**

# Naive solution: Adding a tolerance

Instead of asserting `output == EXPECTED_VALUE`, assert output is _approximately_ the expected value.

In [ ]:
tolerance = EXPECTED_VALUE * 0.1
for i, sim in enumerate(msim.sims):
    print(f"Run {i+1}: {cumulative_infections(sim)} total infections")
    assert (
        (EXPECTED_VALUE - tolerance)
        < cumulative_infections(sim)
        < (EXPECTED_VALUE + tolerance)
    ), f"{cumulative_infections(sim)} not within 10% of {EXPECTED_VALUE}"

Choosing too small a tolerance will lead to failing tests

In [ ]:
tolerance = EXPECTED_VALUE * 0.4
for i, sim in enumerate(msim.sims):
    print(f"Run {i+1}: {sim.results['cum_infections'][-1]:,.0f} total infections")
    assert (
        (EXPECTED_VALUE - tolerance)
        < sim.results["cum_infections"][-1]
        < (EXPECTED_VALUE + tolerance)
    )

To get the test to pass, we need a tollerance of 40%!
- Is this _expected_?
- Is this test _meaningful_?

# What happens with a more infectious variant?

What do we expect to happen?

In [ ]:
msim = cv.MultiSim(cv.Sim(location="UK", beta=0.017, verbose=0))
msim.run(n_runs=5)
for i, sim in enumerate(msim.sims):
    print(f"Run {i+1}: {cumulative_infections(sim)} total infections")

**Are these results what we'd expect?**

# Metamorphic testing

Instead of asserting that `f(x) = y`, assert that _changing_ `x` leads to a _corresponding change in_ `y`.

`beta_1 > beta_2 ==> cum_infections_1 > cum_infections_2`

In [ ]:
beta_1 = 0.016
beta_2 = 0.017

msim_1 = cv.MultiSim(cv.Sim(location="UK", beta=beta_1, verbose=0, pop_type="hybrid"))
msim_2 = cv.MultiSim(cv.Sim(location="UK", beta=beta_2, verbose=0, pop_type="hybrid"))

msim_1.run(n_runs=5)
msim_2.run(n_runs=5)

for sim_1, sim_2 in zip(msim_1.sims, msim_2.sims):
    cum_infections_1 = cumulative_infections(sim_1)
    cum_infections_2 = cumulative_infections(sim_2)
    assert (
        cum_infections_2 > cum_infections_1
    ), f"{cum_infections_2} was less than {cum_infections_1}"

**Does this mean that there's a problem?**

Not necessarily - remember covasim is inherently stochastic

# Statistical metamorphic testing

Instead of testing relationships between exact values, test statistical properties between _populatiuons_ of runs.

In [ ]:
import numpy as np


def test_population_means(n_runs: int):
    msim_1 = cv.MultiSim(
        cv.Sim(location="UK", beta=0.016, verbose=0, pop_type="hybrid")
    )
    msim_2 = cv.MultiSim(
        cv.Sim(location="UK", beta=0.017, verbose=0, pop_type="hybrid")
    )

    msim_1.run(n_runs=n_runs)
    msim_2.run(n_runs=n_runs)

    cum_infections_1 = [cumulative_infections(sim) for sim in msim_1.sims]
    cum_infections_2 = [cumulative_infections(sim) for sim in msim_2.sims]

    assert np.mean(cum_infections_2) > np.mean(
        cum_infections_1
    ), f"{np.mean(cum_infections_2)} < {np.mean(cum_infections_1)}"
    print(f"{np.mean(cum_infections_2)} > {np.mean(cum_infections_1)}")


test_population_means(n_runs=5)

# Is this feasible?

Running with five repeats isn't very statistically significant. Let's try it again with 30 repeats for each.

In [ ]:
from time import time

start = time()
test_population_means(30)
end = time()
total_time = round(end - start, 2)
print(f"That took {total_time} seconds!")

Now let's test every pair of the 202 supported countries...

In [ ]:
from covasim.data.country_age_data import data as countries
from itertools import combinations

country_pairs = list(combinations(countries, 2))

print(f"Covasim has {len(countries)} supported countries.")
print(
    f"That's {len(country_pairs)} combinations - {total_time * len(country_pairs)} seconds!"
)

Covasim has more than 60 parameters, many of which are complex objects with their own sub-parameters.

Collecting the data to test all these relationships could take months!

# Causal testing

Causal testing aleviates this problem by allowing us to re-use data.

The process of data collection (running the model) is completely separate from performing the tests (performing the validation).

The [Causal Testing Framework](https://github.com/CITCOM-project/CausalTestingFramework) provides a suite of tools to facilitate causal testing.

![Causal testing framework workflow](https://github.com/CITCOM-project/CausalTestingFramework/raw/main/images/schematic.png)

# Specify the expected causal relationships

A _directed acyclic graph_ (DAG) shows the expected relationships between model parameters and outputs.7

An edge `X -> Y` represents that `X` causes `Y`, i.e. that the value of `Y` somehow depends on the value of `X`.

In [ ]:
%cat simple_dag.dot

from causal_testing.specification.causal_dag import CausalDAG
from helpers import render_dag

simple_dag = CausalDAG("simple_dag.dot")

render_dag(simple_dag)

# Covasim is actually a little more complex

The location does not determine the cumulative infections directly.

Instead, it determines the average age of the population and the number of contacts each agent has at home, school, work, and in the community.

The user has no direct control over these parameters from outside the model. They can only change the location.

In [ ]:
%cat complex_dag.dot

complex_dag = CausalDAG("complex_dag.dot")
render_dag(complex_dag)

# Collecting data

Run the model a bunch of times with random values for each parameter

In [ ]:
import pandas as pd
import random

random.seed(0)

RUNS = 10
locations = list(cv.data.country_age_data.data)
betas = np.linspace(0.010, 0.020, RUNS)  # Sweep beta from 0.01 to 0.02 with 5 values
msim = cv.MultiSim(
    [
        cv.Sim(
            beta=beta, location=random.choice(locations), pop_type="hybrid", verbose=0
        )
        for beta in betas
    ]
)
msim.run(keep_people=True)
data = []
for sim in msim.sims:
    datum = {
        "location": sim.pars["location"],
        "beta": sim.pars["beta"],
        "average_age": sim.people["age"].mean(),
        "contacts_home": sim.pars["contacts"]["h"],
        "contacts_school": sim.pars["contacts"]["s"],
        "contacts_work": sim.pars["contacts"]["w"],
        "contacts_community": sim.pars["contacts"]["c"],
        "cum_infections": cumulative_infections(sim),
    }
    data.append(datum)
data = pd.DataFrame(data)

# Generating causal tests

In [ ]:
complex_dag.datatypes = data.dtypes
causal_tests = complex_dag.generate_causal_tests()
causal_tests[0].to_dict()

# Running test cases

In [ ]:
from causal_testing.causal_testing_framework import CausalTestingFramework
import warnings

# hide warnings that occur as a result of causal test adequacy calculation
warnings.filterwarnings("ignore")

ctf = CausalTestingFramework(dag=complex_dag, df=data, test_cases=causal_tests)
ctf.run_tests(silent=True, adequacy=True)

# Visualising the results

In [ ]:
from causal_testing.visualisation.causal_test_result_visualiser import results_dag

results = results_dag(dag=ctf.dag, test_cases=ctf.test_cases)

render_dag(results, size="10,10")

[test] = [
    t
    for t in ctf.test_cases
    if t.treatment_variable == "location" and t.outcome_variable == "contacts_home"
]
test.to_dict()

In [ ]:
import hvplot.networkx as hvnx
import networkx as nx
from holoviews import opts
from bokeh.models import HoverTool
from holoviews.operation.datashader import bundle_graph

graphviz_pos = nx.nx_agraph.graphviz_layout(results, prog="dot")

hover = HoverTool(
    tooltips="""
        <div style="padding: 6px; border: 1px solid #ccc; font-family: sans-serif;">
            <strong>Treatment:</strong> @treatment_variable<br>
            <strong>Outcome:</strong> @outcome_variable<br>
            <strong>Causal Effect:</strong> <br/> @title{safe}<br>
        </div>
    """
)

for treatment_variable, outcome_variable in results.edges:
    results[treatment_variable][outcome_variable]["treatment_variable"] = treatment_variable
    results[treatment_variable][outcome_variable]["outcome_variable"] = outcome_variable

causal_tests_graph = hvnx.draw(
    results,
    edgelist=complex_dag.edges,
    pos=graphviz_pos,
    with_labels=True,
    node_color="white",
    edge_color="color",
)
independence_tests_graph = hvnx.draw(
    results,
    edgelist=nx.non_edges(complex_dag),
    pos=graphviz_pos,
    with_labels=True,
    node_color="white",
    edge_color="color",
    style="dashed",
)

(causal_tests_graph * independence_tests_graph).opts(
    opts.Graph(inspection_policy='edges', tools=[hover],),
    
)
# independence_tests

In [ ]:
import holoviews as hv
import networkx as nx
import numpy as np
from bokeh.models import Arrow, ColumnDataSource, NormalHead, Ellipse
from scipy.interpolate import splev, splprep

hv.extension("bokeh")

hover = HoverTool(
    tooltips="""
        <div style="padding: 6px; border: 1px solid #ccc; font-family: sans-serif;">
            <strong>Treatment:</strong> @source<br>
            <strong>Outcome:</strong> @target<br>
            <strong>Causal Effect:</strong> <br/> @title{safe}<br>
        </div>
    """
)

def parse_dot_spline(pos_str):
    """Parse Graphviz 'pos' string into control points."""
    start_pt, end_pt = None, None
    raw_points = []

    for segment in pos_str.split(";"):
        tokens = segment.strip().split(" ")
        for token in tokens:
            if not token:
                continue
            if token.startswith("s,"):
                start_pt = tuple(map(float, token[2:].split(",")))
            elif token.startswith("e,"):
                end_pt = tuple(map(float, token[2:].split(",")))
            else:
                raw_points.append(tuple(map(float, token.split(","))))

    points = []
    if start_pt:
        points.append(start_pt)
    points.extend(raw_points)
    if end_pt:
        points.append(end_pt)

    return points, end_pt


def smooth_spline(pts, num_points=100):
    """Interpolate control points into a high-density smooth curve."""
    pts = np.array(pts)
    if len(pts) < 2:
        return pts

    mask = np.ones(len(pts), dtype=bool)
    mask[1:] = np.any(np.diff(pts, axis=0) != 0, axis=1)
    pts = pts[mask]

    if len(pts) < 4:
        return pts

    tck, _ = splprep([pts[:, 0], pts[:, 1]], k=3, s=0)
    u_new = np.linspace(0, 1, num_points)
    x_new, y_new = splev(u_new, tck)
    return np.column_stack([x_new, y_new])


A = nx.nx_agraph.to_agraph(results)
A.layout(prog="dot")

sample_key = next(iter(results.nodes()))
key_type = type(sample_key)

node_positions = {}
for node in A.nodes():
    x, y = map(float, node.attr["pos"].split(","))
    node_positions[key_type(node.name)] = (x, y)

# Set dynamic width & height per node label
node_ids = list(node_positions.keys())
node_widths = {nid: max(45, len(str(nid)) * 9 + 24) for nid in node_ids}
node_heights = {nid: 32 for nid in node_ids}


# --- 2. Ellipse-Aware Spline Trimming ---
def get_ellipse_radius(angle, a, b):
    """Calculate exact radial distance to ellipse boundary at a given angle."""
    cos_t, sin_t = np.cos(angle), np.sin(angle)
    return (a * b) / np.sqrt((b * cos_t) ** 2 + (a * sin_t) ** 2)


def extend_and_trim_ellipse(
    p_start, raw_pts, p_end, w_start, h_start, w_end, h_end
):
    """Trims spline precisely at source & target ellipse boundaries."""
    pts = [p_start] + raw_pts + [p_end]
    smooth_pts = smooth_spline(pts, num_points=100)

    if len(smooth_pts) < 2:
        return smooth_pts, p_start, p_end

    # Calculate angle and boundary radius for start node
    dx_s = smooth_pts[5][0] - p_start[0]
    dy_s = smooth_pts[5][1] - p_start[1]
    angle_start = np.arctan2(dy_s, dx_s)
    r_start = get_ellipse_radius(
        angle_start, w_start / 2.0, h_start / 2.0
    )

    dists_start = np.hypot(
        smooth_pts[:, 0] - p_start[0], smooth_pts[:, 1] - p_start[1]
    )
    start_idx = np.searchsorted(dists_start, r_start)
    start_idx = min(start_idx, len(smooth_pts) // 3)

    # Calculate angle and boundary radius for end node
    dx_e = p_end[0] - smooth_pts[-5][0]
    dy_e = p_end[1] - smooth_pts[-5][1]
    angle_end = np.arctan2(dy_e, dx_e)
    r_end = get_ellipse_radius(angle_end, w_end / 2.0, h_end / 2.0)

    dists_end = np.hypot(
        smooth_pts[:, 0] - p_end[0], smooth_pts[:, 1] - p_end[1]
    )
    end_idx = len(smooth_pts) - np.searchsorted(dists_end[::-1], r_end)
    end_idx = max(end_idx, start_idx + 5)

    trimmed_path = smooth_pts[start_idx:end_idx]
    return trimmed_path, trimmed_path[-3], trimmed_path[-1]


# --- 3. Process Edge Data ---
edge_rows = []
for u, v, data in results.edges(data=True):
    row = {"source": u, "target": v}
    row.update(data)
    edge_rows.append(row)

edges_df = pd.DataFrame(edge_rows)
edge_vdims = [c for c in edges_df.columns if c not in ["source", "target"]]

edge_paths = []
arrow_starts_x, arrow_starts_y = [], []
arrow_ends_x, arrow_ends_y = [], []
arrow_colors = []
edge_label_data = []

for i, row in edges_df.iterrows():
    u, v = row["source"], row["target"]
    p_start, p_end = node_positions[u], node_positions[v]
    edge_color = row.get("color", "gray")

    edge = A.get_edge(str(u), str(v))
    pos_str = edge.attr["pos"]
    raw_pts, _ = parse_dot_spline(pos_str)

    trimmed_path, p_near_tip, p_tip = extend_and_trim_ellipse(
        p_start,
        raw_pts,
        p_end,
        node_widths[u],
        node_heights[u],
        node_widths[v],
        node_heights[v],
    )

    edge_paths.append(trimmed_path)
    mid_pt = trimmed_path[len(trimmed_path) // 2]
    edge_label_data.append(
        (mid_pt[0], mid_pt[1], str(row.get("label", "")))
    )

    arrow_starts_x.append(p_near_tip[0])
    arrow_starts_y.append(p_near_tip[1])
    arrow_ends_x.append(p_tip[0])
    arrow_ends_y.append(p_tip[1])
    arrow_colors.append(edge_color)

# --- 4. Build Datasets ---
nodes_ds = hv.Nodes(
    [(x, y, node_id) for node_id, (x, y) in node_positions.items()],
    kdims=["x", "y", "index"],
)
edges_ds = hv.EdgePaths(edge_paths)

graph = hv.Graph(
    (edges_df, nodes_ds, edges_ds),
    kdims=["source", "target"],
    vdims=edge_vdims,
)

# Label Layers
node_label_data = [
    (x, y, str(node_id)) for node_id, (x, y) in node_positions.items()
]
node_labels = hv.Labels(
    node_label_data, kdims=["x", "y"], vdims=["label"]
).opts(
    text_font_size="9pt",
    text_color="black",
    text_align="center",
    text_baseline="middle",
    yoffset=0,
)

edge_labels = hv.Labels(
    edge_label_data, kdims=["x", "y"], vdims=["label"]
).opts(
    text_font_size="8pt",
    text_color="darkblue",
    text_align="center",
    text_baseline="middle",
)


# --- 5. Custom Hook for Hover/Selection Glyph Override & Colored Arrows ---
def style_graph_hook(plot, element):
    fig = plot.handles["plot"]
    graph_renderer = plot.handles["glyph_renderer"]

    # Supply widths and heights to the node source
    node_source = graph_renderer.node_renderer.data_source
    node_source.data["width"] = [node_widths[nid] for nid in node_ids]
    node_source.data["height"] = [node_heights[nid] for nid in node_ids]

    # Define primary Ellipse glyph
    ellipse_glyph = Ellipse(
        x="x",
        y="y",
        width="width",
        height="height",
        fill_color="white",
        line_color="gray",
    )

    # Define hover / inspection Ellipse glyph (prevents reverting to green circles)
    hover_ellipse = Ellipse(
        x="x",
        y="y",
        width="width",
        height="height",
        fill_color="skyblue",
        line_color="gray",
    )

    # Override all state glyphs so nodes remain ellipses during inspection
    graph_renderer.node_renderer.glyph = ellipse_glyph
    graph_renderer.node_renderer.hover_glyph = hover_ellipse
    graph_renderer.node_renderer.selection_glyph = ellipse_glyph
    graph_renderer.node_renderer.nonselection_glyph = ellipse_glyph

    # Add Arrowheads with matching edge colors
    for i in range(len(arrow_starts_x)):
        color = arrow_colors[i]
        head = NormalHead(fill_color=color, line_color=color, size=8)
        arrow = Arrow(
            end=head,
            x_start=arrow_starts_x[i],
            y_start=arrow_starts_y[i],
            x_end=arrow_ends_x[i],
            y_end=arrow_ends_y[i],
            line_alpha=0,
        )
        fig.add_layout(arrow)


# --- 8. Style & Render Overlay ---
graph.opts(
    edge_line_dash="style",
    edge_line_width=1.5,
    edge_color="color",
    width=900,
    height=450,
    hooks=[style_graph_hook],
    tools=[hover],
    inspection_policy='edges',
    edge_hover_line_color="color",  # Retains original edge color during hover
)

final_plot = graph * node_labels * edge_labels
final_plot

# Causal Test Adequacy

Just because a test has failed doesn't mean there's a problem with the model! There could be a problem with our causal DAG or we may not have enough data. Causal Test Adequacy tells us how trustworthy our causal test outcomes are.

In [ ]:
from helpers import data_adequacy_heatmap

data_adequacy_heatmap(ctf.test_cases)

In short:

- Values close to 0 indicate trustworthy estimates
- Values below zero are "suspiciously deterministic"
- Values above zero are unstable - extra data is needed

For further details, [read the paper](https://ieeexplore.ieee.org/document/10638595).

# Evaluating causal DAGs

We can see how well a given causal DAG fits the data we have, and estimate the confidence that it's accurate.

To do this, we repeatedly resample our data and execute our causal tests on the samples.

In [ ]:
from helpers import dag_adequacy_heatmap

dag_adequacy_heatmap(ctf.test_cases)

# Inferring causal DAGs

We can also infer causal DAGs from the data.

In [ ]:
from causal_testing.discovery.hill_climber_discovery import HillClimberDiscovery

hill_climber = HillClimberDiscovery(df=data)
discovered_dag = hill_climber.discover()
render_dag(discovered_dag)

This DAG doesn't make much sense:

- `location` and `beta` are inputs, so are independent of the other variables
- `cum_infections` is an output, so cannot cause anything else

Let's add this knowledge to the discovery

In [ ]:
hill_climber = HillClimberDiscovery(
    df=data,
    exclude_edges=[(".*", "beta"), (".*", "location"), ("cum_infections", ".*")],
)
discovered_dag = hill_climber.discover()
render_dag(discovered_dag)

In [ ]:
discovered_ctf = CausalTestingFramework(
    dag=discovered_dag, df=data, test_cases=discovered_dag.generate_causal_tests()
)
discovered_ctf.run_tests(silent=True, adequacy=True)

In [ ]:
data_adequacy_heatmap(discovered_ctf.test_cases)

In [ ]:
dag_adequacy_heatmap(discovered_ctf.test_cases)

# Conclusion

- Scientific software is inherently hard to test
- Instead of asserting that a particular input configuration results in a particular output configuration, we can test the _relationships_ between inputs and outputs.
- Collecting repeated test runs for every parameter configuration we want to test can be infeasible.
- Causal testing allows us to maximise what we can do with the test runs we are able to collect.
- The Causal Testing Framework automates much of this process.